In [1]:
import pandas as pd
import numpy as np
import math
import warnings

import gseapy as gp

def addColumns(data):
    # Responders: CR/PR, Non responder: PD
    responderCrit = data['BR'].isin(['CR','PR'])
    progressorCrit = data['BR'].isin(['PD'])
    
    data["PD_Responders"] = "SD/MR"
    responders = data[responderCrit].index
    progressors = data[progressorCrit].index
    nonresponders = data[~responderCrit].index
    nonprogressors = data[~progressorCrit].index
    
    data.loc[responders,"PD_Responders"] = 'Responder'
    data.loc[progressors,"PD_Responders"] = 'Progressor'
    return data

def load_gmt_as_dict(gmt_file):
    gene_sets = {}
    with open(gmt_file, "r") as f:
        for line in f:
            parts = line.strip().split("\t")
            if len(parts) < 3:
                continue
            set_name = parts[0]
            # parts[1] is description (often ignored)
            # Upper-case genes and remove any empty strings / duplicates
            genes = [g.strip().upper() for g in parts[2:] if g.strip()]
            # preserve order but remove duplicates
            seen = set()
            genes_unique = []
            for g in genes:
                if g not in seen:
                    seen.add(g)
                    genes_unique.append(g)
            gene_sets[set_name] = genes_unique
    return gene_sets

mhc_ii = [
    "HLA-DMA", "HLA-DMB", "HLA-DOA", "HLA-DOB", "HLA-DPA1", "HLA-DPB1",
    "HLA-DQA1", "HLA-DQA2", "HLA-DQB1", "HLA-DQB2", "HLA-DRA", "HLA-DRB1", "HLA-DRB5"
]
mhc_i = ["HLA-A", "HLA-B", "HLA-C", "HLA-E", "HLA-F", "TAP1", "TAP2", "B2M"]

hallmarks_gmt = "h.all.v2025.1.Hs.symbols.gmt"
gmt_dict = load_gmt_as_dict(hallmarks_gmt)  # returns dict with upper-cased genes

# add custom sets as upper-cased lists
gmt_dict["MHC-II"] = [g.upper() for g in mhc_ii]
gmt_dict["MHC-I"] = [g.upper() for g in mhc_i]



In [3]:
seeds = [0,1,2,3]
for seed in seeds:
    print(f'****{seed}****')
    syn_datas = ["original_data",f'avatarsk5_{seed}',f"avatarsk10_{seed}",f"ctgan_{seed}",f"gaussiancopula_{seed}",f"synthpop_{seed}",f"tvae_{seed}"]
    dataset_dict = {}
    for syn_data in syn_datas:
        print(f"-----{syn_data}-----")
        df = pd.read_csv(f"../../Data/{syn_data}.csv",index_col = 0)
        processed_df = addColumns(df)
        dataset_dict[syn_data] = processed_df
    
    ssGSEA = {}
    data = dataset_dict['original_data']
    genes_cols = data.columns.tolist()[54:-1]
    for tool, data in dataset_dict.items():
        ssgsea_res = gp.ssgsea(
            data=data[genes_cols].T,
            gene_sets=gmt_dict,
            sample_norm_method='rank',
            outdir=None,
            permutation_num=0,
            no_plot=True,
            min_size=1,
            max_size=10000
        )
        res2d = ssgsea_res.res2d
        
        res_wide = res2d.pivot(index="Name", columns="Term", values="ES")
        res_wide.to_csv(f"MHC/{tool}.csv", index = True)
        ssGSEA[tool] = res_wide

****0****
-----original_data-----
-----avatarsk5_0-----
-----avatarsk10_0-----
-----ctgan_0-----
-----gaussiancopula_0-----
-----synthpop_0-----
-----tvae_0-----
****1****
-----original_data-----
-----avatarsk5_1-----
-----avatarsk10_1-----
-----ctgan_1-----
-----gaussiancopula_1-----
-----synthpop_1-----
-----tvae_1-----
****2****
-----original_data-----
-----avatarsk5_2-----
-----avatarsk10_2-----
-----ctgan_2-----
-----gaussiancopula_2-----
-----synthpop_2-----
-----tvae_2-----
****3****
-----original_data-----
-----avatarsk5_3-----
-----avatarsk10_3-----
-----ctgan_3-----
-----gaussiancopula_3-----
-----synthpop_3-----
-----tvae_3-----
